In [1]:
import json
import re
from pathlib import Path
from copy import deepcopy


INPUT_PATH = "/home/vcnt/Repositories/Tesis/datasets_metadata/histai_colorectal_b1_metadata.json"
OUTPUT_PATH = "/home/vcnt/Repositories/Tesis/datasets_metadata/histai_colorectal_b1_metadata_regroup.json"

#Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

def normalize_text(s):
    if s is None:
        return ""
    s = str(s).strip().lower()
    s = s.replace("c-r", "cancer")
    s = s.replace("cr ", "cancer ")
    s = s.replace("crc", "colorectal cancer")
    s = s.replace("ca ", "cancer ")
    s = re.sub(r'c18\.0', 'cecal cancer', s)
    s = re.sub(r'c18\.1', 'appendiceal cancer', s)
    s = re.sub(r'c18\.2', 'ascending colon cancer', s)
    s = re.sub(r'c18\.3', 'hepatic flexure colon cancer', s)
    s = re.sub(r'c18\.4', 'transverse colon cancer', s)
    s = re.sub(r'c18\.5', 'splenic flexure colon cancer', s)
    s = re.sub(r'c18\.6', 'descending colon cancer', s)
    s = re.sub(r'c18\.7', 'sigmoid colon cancer', s)
    s = re.sub(r'c19', 'rectosigmoid cancer', s)
    s = re.sub(r'c20', 'rectal cancer', s)
    s = re.sub(r'd12\.5', 'sigmoid colon adenoma', s)
    s = re.sub(r'd37\.7', 'intestinal neoplasm uncertain', s)
    s = re.sub(r'd 37\.7', 'intestinal neoplasm uncertain', s)
    s = re.sub(r'[^a-z0-9\s\?\-]', ' ', s)
    s = re.sub(r'\s+', ' ', s).strip()
    return s

def detect_site(s):
    site_rules = [
        ("rectosigmoid_junction", [r"rectosigmoid junction", r"rectosigmoid"]),
        ("rectum", [r"\brectal\b", r"\brectum\b", r"rectal ampulla", r"ampullary", r"ampulla"]),
        ("sigmoid_colon", [r"sigmoid colon", r"\bsigmoid\b", r"\bsigma\b"]),
        ("descending_colon", [r"descending colon"]),
        ("splenic_flexure", [r"splenic flexure"]),
        ("transverse_colon", [r"transverse colon"]),
        ("hepatic_flexure", [r"hepatic flexure"]),
        ("ascending_colon", [r"ascending colon"]),
        ("cecum", [r"\bcecal\b", r"\bcecum\b", r"cecal dome", r"ileocecal"]),
        ("anal_canal", [r"anal canal", r"\banal\b", r"internal anal sphincter"]),
        ("colon_unspecified", [r"\bcolon\b", r"\bcolonic\b", r"large intestine", r"\bcolorectal\b", r"\bintestinal\b", r"\bbowel\b"]),
    ]
    for label, patterns in site_rules:
        if any(re.search(p, s) for p in patterns):
            return label
    return "unspecified"

def has_uncertainty(s):
    terms = ["?", "suspicious", "suspicion", "suspected", "possible", "likely", "uncertain"]
    return any(t in s for t in terms)

def classify_diagnosis(raw_dx):
    s = normalize_text(raw_dx)
    site = detect_site(s)

    if s in {"", "-", "examination", "observation", "gastrointestinal tract examination", "colorectal disease"}:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "missing_or_uninformative",
            "taxonomy_subgroup": "missing_or_unspecified",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "uninformative"
        }

    if any(x in s for x in ["stomach", "gastric", "gastritis", "esophagitis", "esophageal", "prostate cancer", "cholecystitis", "cholelithiasis", "peritonitis"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "non_colorectal_or_noise",
            "taxonomy_subgroup": "other_organ_or_noise",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "non_colorectal"
        }

    if any(x in s for x in ["ulcerative colitis", "colitis", "proctitis", "ileitis", "proctosigmoiditis", "typhlitis", "ischemic catarrhal colitis", "inflammation"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "inflammatory_or_non_neoplastic",
            "taxonomy_subgroup": "colitis_or_proctitis",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "inflammatory"
        }

    if any(x in s for x in ["hyperplasia", "lymphofollicular hyperplasia", "infiltrate"]) and "adenocarcinoma" not in s:
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "inflammatory_or_non_neoplastic",
            "taxonomy_subgroup": "hyperplasia_or_non_neoplastic_lesion",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "hyperplasia"
        }

    if any(x in s for x in ["metastatic colorectal cancer", "with metastases", "secondary malignant neoplasm of the liver", "metastatic"]) and any(y in s for y in ["colon", "rect", "sigmoid", "colorectal", "cecal", "adenocarcinoma", "cancer"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "metastatic_colorectal_cancer",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "metastatic_crc"
        }

    if any(x in s for x in ["neuroendocrine tumor", "carcinoid", "lymphoma"]) and any(y in s for y in ["rect", "colon", "cecal", "anal", "intestinal"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "neuroendocrine_or_carcinoid",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "neuroendocrine"
        }

    if any(x in s for x in ["adenocarcinoma", "moderately differentiated adenocarcinoma", "poorly differentiated adenocarcinoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "adenocarcinoma",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "adenocarcinoma"
        }

    if any(x in s for x in ["cancer", "malignant neoplasm", "carcinoma", "blastoma"]) and not has_uncertainty(s):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "malignant",
            "taxonomy_subgroup": "colorectal_carcinoma_nos",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "crc_nos"
        }

    if has_uncertainty(s) and any(x in s for x in ["cancer", "neoplasm", "tumor", "carcinoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "suspicious_or_uncertain",
            "taxonomy_subgroup": "suspicious_for_malignancy",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "suspicious_malignancy"
        }

    if any(x in s for x in ["tubulovillous adenoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "tubulovillous_adenoma",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "tubulovillous_adenoma"
        }

    if any(x in s for x in ["tubular adenoma"]):
        subgroup = "tubular_adenoma" if not has_uncertainty(s) else "adenoma"
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": subgroup,
            "taxonomy_site": site,
            "taxonomy_status": "review" if has_uncertainty(s) else "final",
            "taxonomy_rule": "tubular_adenoma"
        }

    if any(x in s for x in ["villous tumor", "villous adenoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "villous_adenoma_or_tumor",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "villous_lesion"
        }

    if any(x in s for x in ["serrated lesion", "serrated adenoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "serrated_lesion_or_adenoma",
            "taxonomy_site": site,
            "taxonomy_status": "review" if has_uncertainty(s) else "final",
            "taxonomy_rule": "serrated"
        }

    if any(x in s for x in ["hyperplastic polyp"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "hyperplastic_polyp",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "hyperplastic_polyp"
        }

    if any(x in s for x in ["adenoma", "benign neoplasm"]) and not any(x in s for x in ["adenocarcinoma"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "adenoma",
            "taxonomy_site": site,
            "taxonomy_status": "final",
            "taxonomy_rule": "adenoma"
        }

    if any(x in s for x in ["polyp", "polyps", "polypoid"]) and not any(x in s for x in ["adenocarcinoma", "cancer"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "benign_precursor",
            "taxonomy_subgroup": "polyp_nos",
            "taxonomy_site": site,
            "taxonomy_status": "review" if has_uncertainty(s) else "final",
            "taxonomy_rule": "polyp"
        }

    if any(x in s for x in ["epithelial formation", "epithelial formations", "epithelial neoplasm", "epithelial lesion", "formation", "neoplasm", "tumor", "mass", "lesion", "neoplasia"]):
        return {
            "diagnosis_normalized": s,
            "taxonomy_group": "nonspecific_neoplastic",
            "taxonomy_subgroup": "epithelial_formation_or_neoplasm",
            "taxonomy_site": site,
            "taxonomy_status": "review",
            "taxonomy_rule": "nonspecific_neoplastic"
        }

    return {
        "diagnosis_normalized": s,
        "taxonomy_group": "non_colorectal_or_noise",
        "taxonomy_subgroup": "unmapped_review",
        "taxonomy_site": site,
        "taxonomy_status": "review",
        "taxonomy_rule": "fallback"
    }

with open(INPUT_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

out = []
for row in data:
    row2 = deepcopy(row)
    mapped = classify_diagnosis(row.get("diagnosis", ""))
    row2.update(mapped)
    out.append(row2)

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(out, f, ensure_ascii=False, indent=2)

print(f"Saved {len(out)} records to {OUTPUT_PATH}")

Saved 878 records to /home/vcnt/Repositories/Tesis/datasets_metadata/histai_colorectal_b1_metadata_regroup.json
